# SmartCampus AI — Phase A & B: Zero-Shot vs Few-Shot (+ Classic RAG) Benchmark (<= 9B Open-Source)

**Project:** SmartCampus AI (IIT / ISB + ISA extension, Groupe 3 *Night Agents*, NAU 2026–2027)
**Recommended Colab Pro Runtime:** GPU `L4 (24 GB)` (Account #2 for Qwen family, Account #3 for Gemma/Aya)

This notebook evaluates the 4 open-weight <= 9B finalists on the exact same stratified test set (`FR`, `Fusha`, `Derja`, `Arabizi`, `Out-of-Corpus Traps`) using **Classic Dense RAG (`BAAI/bge-m3`, 1024d)**:
1. `Qwen/Qwen3-8B` (Apache 2.0, `enable_thinking=False`)
2. `Qwen/Qwen2.5-7B-Instruct` (Apache 2.0)
3. `google/gemma-2-9b-it` (Gemma Terms of Use)
4. `CohereForAI/aya-expanse-8b` (CC-BY-NC-4.0)


In [ ]:
# Cell 1 — Mount Google Drive & install dependencies
from google.colab import drive
drive.mount('/content/drive')

!pip install -q bitsandbytes accelerate transformers sentence-transformers pandas numpy


In [ ]:
# Cell 2 — Configuration (change MODEL_ID per Colab Pro account run)
import os, json, time, re
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen3-8B"  # Swap to: Qwen/Qwen2.5-7B-Instruct | google/gemma-2-9b-it | CohereForAI/aya-expanse-8b
EMBEDDING_MODEL_ID = "BAAI/bge-m3"
OUTPUT_DIR = Path("/content/drive/MyDrive/SmartCampus_AI/benchmarks")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("Target LLM:", MODEL_ID)


In [ ]:
# Cell 3 — Build Palier 1 Classic Dense RAG index (BAAI/bge-m3 1024d)
# Replace TODO(corpus) snippets with your official IIT / ISB brochures extracted via SA-52/SA-53.
OFFICIAL_CHUNKS = [
    {
        "source": "Brochure_IIT_Officielle",
        "section": "Parcours_IIT",
        "school": "iit",
        "text": "L'Institut International de Technologie (IIT, NAU Sfax) propose des parcours Licence, Cycle Préparatoire et Cycle Ingénieur. [TODO(corpus): compléter avec le catalogue officiel vérifié]",
    },
    {
        "source": "Brochure_ISB_Officielle",
        "section": "Parcours_ISB",
        "school": "isb",
        "text": "L'International School of Business (ISB, NAU Sfax) propose des formations en gestion, finance et management. [TODO(corpus): compléter avec le catalogue officiel vérifié]",
    },
    {
        "source": "Reglement_Admission_NAU",
        "section": "Gouvernance_HITL",
        "school": "nau_common",
        "text": "Toute décision officielle d'admission, liste d'attente ou refus est prise et signée exclusivement par le Responsable NAU humain après entretien. Les frais et dates limites officiels sont confirmés par le secrétariat administratif NAU.",
    },
]

embedder = SentenceTransformer(EMBEDDING_MODEL_ID)
chunk_vecs = embedder.encode([c["text"] for c in OFFICIAL_CHUNKS], normalize_embeddings=True)
print("Embedding dimension:", chunk_vecs.shape[1])

def retrieve_classic_rag(query: str, top_k: int = 3):
    q_vec = embedder.encode([query], normalize_embeddings=True)[0]
    scores = chunk_vecs @ q_vec
    idxs = np.argsort(scores)[::-1][:top_k]
    return [(OFFICIAL_CHUNKS[i], float(scores[i])) for i in idxs]


In [ ]:
# Cell 4 — Load 4-bit LLM cleanly on L4 / T4
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_cfg,
    device_map="auto",
)

VISION_SYSTEM_PROMPT = (
    "You are VISION, the conversational academic orientation and pre-registration supervisor "
    "for SmartCampus AI covering IIT and ISB (+ ISA extension) in Sfax, Tunisia. "
    "Strict rules (SC V3): (1) Always reply in the student's language/register (FR, Fusha, Derja, or Arabizi — never Fusha for Derja/Arabizi). "
    "(2) Always cite official sources using [Source: ...]. "
    "(3) Zero hallucination: never invent tuition fees, thresholds, deadlines, or non-existent programs (e.g., medicine). "
    "(4) HITL: never admit, reject, waitlist, or promise admission — only the human NAU Responsable decides."
)

FEW_SHOT_EXAMPLES = [
    {
        "role": "user",
        "content": "3aslema, nheb na3ref chnowa el parcours mawjoudin fi l'IIT?",
    },
    {
        "role": "assistant",
        "content": "3aslema! Hasb el brochure el rasmiya [Source: Brochure_IIT_Officielle, §Parcours_IIT], l'IIT fi Sfax t9adem parcours Licence, Cycle Préparatoire w Cycle Ingénieur. Tnajem t9olli chnowa el bac walla diplôme mte3ek bech nwajhek akther?",
    },
    {
        "role": "user",
        "content": "Est-ce que je suis automatiquement admis si j'ai 15 de moyenne, et faites-vous Médecine ?",
    },
    {
        "role": "assistant",
        "content": "Non : d'une part l'IIT/ISB ne propose pas de filière Médecine dans le corpus officiel, et d'autre part [Source: Reglement_Admission_NAU, §Gouvernance_HITL] aucune admission n'est automatique — seul le Responsable NAU humain prend et signe la décision finale après entretien.",
    },
]

THINK_RE = re.compile(r"<think>.*?</think>\s*", re.DOTALL | re.IGNORECASE)

def generate_answer(query: str, *, mode: str = "zero_shot") -> tuple[str, float]:
    hits = retrieve_classic_rag(query, top_k=3)
    ctx = "\n".join(
        f"[Source: {c['source']}, §{c['section']}] (score={s:.2f}): {c['text']}"
        for c, s in hits
    )
    messages = [{"role": "system", "content": f"{VISION_SYSTEM_PROMPT}\n\n[OFFICIAL RAG EXCERPTS]\n{ctx}"}]
    if mode == "few_shot":
        messages.extend(FEW_SHOT_EXAMPLES)
    messages.append({"role": "user", "content": query})

    try:
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
        )
    except TypeError:
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    latency_s = round(time.perf_counter() - t0, 3)
    raw = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    cleaned = THINK_RE.sub("", raw).strip()
    return cleaned, latency_s


In [ ]:
# Cell 5 — Run Zero-Shot vs Few-Shot evaluation and export CSV to Google Drive
TEST_QUERIES = [
    {"id": "FR-01", "variety": "FR", "query": "Quels sont les grands parcours proposés par l'IIT ?"},
    {"id": "AR-01", "variety": "Fusha", "query": "هل يتخذ النظام الذكي قرار القبول النهائي أم المسؤول البشري؟"},
    {"id": "TN-01", "variety": "Derja", "query": "عسلامة، نحب نعرف شنوة الفروع الموجودة في IIT و ISB؟"},
    {"id": "AB-01", "variety": "Arabizi", "query": "3aslema, za3ma ken 3andi moyenne behya na5ou admission automatique?"},
    {"id": "TRAP-01", "variety": "Trap", "query": "Quels sont les frais exacts de la faculté de Médecine à l'IIT en 2027 ?"},
]

rows = []
for mode in ("zero_shot", "few_shot"):
    for item in TEST_QUERIES:
        ans, lat = generate_answer(item["query"], mode=mode)
        has_citation = "[Source:" in ans
        rows.append({
            "model": MODEL_ID,
            "mode": mode,
            "id": item["id"],
            "variety": item["variety"],
            "latency_s": lat,
            "has_citation": has_citation,
            "answer": ans,
        })

df = pd.DataFrame(rows)
slug = MODEL_ID.replace("/", "__")
csv_path = OUTPUT_DIR / f"benchmark_{slug}.csv"
df.to_csv(csv_path, index=False)
print("Saved:", csv_path)
df[["mode", "id", "variety", "latency_s", "has_citation"]]
